# 05 — Fetch Subscriptions

Pulls **every** subscription from MySQL (no `AccountCode` filter — this
loads the full set to be split across the two target accounts), then for
each one:

1. Decides which target account it belongs to (`resolve_target_account`,
   based on `Reference`).
2. Looks up its real address + radius username from Voyager
   (`get_voyager_address`) — `SupplierServiceID` -> `GET .../fibre/v1/circuits/{id}`
   (gives `radiusUsers[0]` + `locationId`) -> `GET .../address-search/v3/addresses/id/{locationId}`
   (gives the actual street address, city, postcode, region). Region name
   is mapped to a 3-char ISO code via `NZ_Regions.xlsx`.

This is a data-prep step that calls out to Voyager (not OneBill) — no
OneBill API calls happen here. Output feeds both `06_Create_Addresses.ipynb`
and `07_Create_Subscription_Orders.ipynb` (the latter now uses the resolved
`radius_user` instead of `SubscriptionLabel`).

> **TODO**: confirm the `Reference` column name
> (`SUBSCRIPTION_REFERENCE_COLUMN` in `onebill_common.py`, currently
> `"Reference"`).

## 1. Setup

In [16]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from sqlalchemy import create_engine
from concurrent.futures import ThreadPoolExecutor

logger = get_logger("fetch_subscriptions")

# Use this to limit rows while testing. Set to None once ready for a full run.
TEST_ROW_LIMIT = None
BATCH_NUMBER =  os.environ["BATCH_NUMBER"]

## 2. Pull every subscription from MySQL

In [17]:
assert BI_DATASTORE_URL, "DB_USERNAME/DB_PASSWORD/DB_HOST not set in .env"
engine = create_engine(BI_DATASTORE_URL)

SUBSCRIPTION_QUERY = '''
SELECT 
    *
FROM bi_datastore.billing_subscription
WHERE _DataSource = 'vBill'
AND AccountCode = '99965692'
AND SubscriptionEndDate IS NULL;
'''.strip()

df_subscriptions = pd.read_sql(SUBSCRIPTION_QUERY, con=engine)
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from MySQL")

if TEST_ROW_LIMIT is not None:
    df_subscriptions = df_subscriptions.head(TEST_ROW_LIMIT)  # Testing limiter — remove/raise for a full run.
    logger.info(f"TEST_ROW_LIMIT active — trimmed to {len(df_subscriptions):,} rows")

df_subscriptions.head()


2026-07-27 10:11:06,824 [INFO] Loaded 264 subscriptions from MySQL


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,CircuitType,Server,CustomerSuppliedReference,_notforreports_VoyagerOrderHistory,_notforreports_LegacyServiceDescription,NextPlanCode,NextPlanStartDate,NextQuantity,NextCustomPrice,SalesAgentCode
0,4163350815,2026-07-03 20:36:41,260726,vBill,99965692,Broadband - Fibre,V113061451,matthew.horncastle@williamsinternet.com,2025-11-26,None,...,UFB 1000/500/2.5/2.5,None,Williams Corporation,CPP 1006617,None,None,None,None,None,None
1,4167925657,2025-12-02 01:12:40,260841,vBill,99965692,Platforms,V113062335,V113062335,2025-12-01,None,...,None,None,None,CAS-973970-B7G2L8,None,None,None,None,None,None
2,4167926822,2025-12-02 01:12:40,260842,vBill,99965692,Platforms,V113062343,V113062343,2025-12-01,None,...,None,None,None,CAS-973970-B7G2L8,None,None,None,None,None,None
3,4167997507,2026-07-03 20:36:54,260846,vBill,99965692,Broadband - Fibre,V113062384,9matataway@williamsinternet.com,2025-12-01,None,...,UFB 100/20/2.5/2.5,None,WC CHCH T9,CPP 2006221,None,None,None,None,None,None
4,4167998688,2026-07-03 20:36:54,260847,vBill,99965692,Broadband - Fibre,V113062392,3matataway@williamsinternet.com,2025-12-01,None,...,UFB 100/20/2.5/2.5,None,WC CHCH T9,CPP 2006218,None,None,None,None,None,None


## 2b. Split active / inactive

Inactive subscriptions (`SubscriptionEndDate` already in the past) go
through a completely separate, much simpler pipeline —
`05_Fetch_Inactive_Subscriptions.ipynb` -> `06_Attach_Inactive_Addresses.ipynb`
-> `07_Create_Inactive_Subscription_Orders.ipynb` — no Voyager lookup at all
(the circuits endpoint doesn't return anything for a subscription that's
already ended), and no new address (they attach to the account's existing
default service address instead).

This notebook, from here on, only ever deals with ACTIVE subscriptions.
The inactive subset is saved to disk here so the inactive pipeline doesn't
need to re-run the same MySQL query.


In [18]:
_end_date = pd.to_datetime(df_subscriptions["SubscriptionEndDate"], errors="coerce")
_today = pd.Timestamp.now().normalize()
_is_active = _end_date.isna() | (_end_date >= _today)

df_subscriptions_inactive_raw = df_subscriptions[~_is_active].copy()
df_subscriptions = df_subscriptions[_is_active].copy()

logger.info(
    f"{len(df_subscriptions):,} active / {len(df_subscriptions_inactive_raw):,} inactive "
    f"(inactive = SubscriptionEndDate before {_today.date()})"
)

if not df_subscriptions_inactive_raw.empty:
    save_df("subscriptions_inactive_raw", df_subscriptions_inactive_raw)
    logger.info(
        f"Saved {len(df_subscriptions_inactive_raw):,} inactive subscriptions to "
        f"'subscriptions_inactive_raw' — pick these up in 05_Fetch_Inactive_Subscriptions.ipynb"
    )


2026-07-27 10:11:06,858 [INFO] 264 active / 0 inactive (inactive = SubscriptionEndDate before 2026-07-27)


## 3. Resolve target account for every subscription

`TargetAccountNumber` is now primarily the subscription's **own** account —
matched by `AccountCode` against `04_Create_Accounts.ipynb`'s real results
(`load_account_code_batch_map`), which is the actual OneBill `accountNumber`
(`AccountCode` + `.` + `BATCH_NUMBER`) for that account. This is what the
account is really called inside OneBill, so subscriptions land back on their
own account instead of one of the two shared Williams buckets.

If a subscription's own account has no successful (`created`/`exists`) row
in `account_results` — e.g. it wasn't migrated, or failed — it falls back to
the old two-bucket routing (`resolve_target_account`, based on `Reference`)
so the row can still be processed rather than being dropped outright.

Run `04_Create_Accounts.ipynb` before this notebook if you haven't already.

In [19]:
own_account_map = load_account_code_batch_map()  # {AccountCode: AccountCode_Batch}, every account in 04's results
real_account_numbers = load_real_target_account_numbers()  # {"managed_by_williams": "...", "williams_corporation": "..."}

missing_keys = set(TARGET_ACCOUNTS) - set(real_account_numbers)
if missing_keys:
    logger.warning(
        f"No successful account_results row found for: {missing_keys} — "
        f"falling back to the TARGET_ACCOUNTS placeholder for those. "
        f"Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run."
    )

logger.info(f"{len(own_account_map):,} accounts available from 04_Create_Accounts.ipynb (own-account routing)")

reference_col = SUBSCRIPTION_REFERENCE_COLUMN if SUBSCRIPTION_REFERENCE_COLUMN in df_subscriptions.columns else None
if reference_col is None:
    logger.warning(
        f"Column '{SUBSCRIPTION_REFERENCE_COLUMN}' not found in df_subscriptions — "
        f"Managed-by-Williams routing and the bucket fallback below can't work without it. "
        f"Available columns: {list(df_subscriptions.columns)}"
    )
    df_subscriptions["CustomerSuppliedReference"] = None
else:
    df_subscriptions["CustomerSuppliedReference"] = df_subscriptions[reference_col]

df_subscriptions["AccountCode"] = df_subscriptions["AccountCode"].astype(str)

# --- Business decision (2026-07-23): "Managed by Williams" subscriptions ALWAYS
# route to the shared bucket account, even when the subscription's own individual
# OneBill account already exists (created by 04_Create_Accounts.ipynb for every
# AccountCode). This takes priority over own-account routing below — it is
# checked first, not used as a fallback.
is_managed_by_williams = (
    df_subscriptions["CustomerSuppliedReference"].fillna("").astype(str).str.lower()
    .str.contains(MANAGED_BY_WILLIAMS_MARKER, regex=False)
)

managed_by_williams_number = real_account_numbers.get(
    "managed_by_williams", TARGET_ACCOUNTS["managed_by_williams"]["account_number"]
)
williams_corporation_number = real_account_numbers.get(
    "williams_corporation", TARGET_ACCOUNTS["williams_corporation"]["account_number"]
)

df_subscriptions["TargetAccountKey"] = None
df_subscriptions["TargetAccountNumber"] = None

# 1. Managed by Williams — always wins, regardless of own account.
df_subscriptions.loc[is_managed_by_williams, "TargetAccountKey"] = "managed_by_williams"
df_subscriptions.loc[is_managed_by_williams, "TargetAccountNumber"] = managed_by_williams_number
logger.info(
    f"{is_managed_by_williams.sum():,} subscriptions routed to the shared 'Managed by Williams' "
    f"account (Reference match — overrides own-account routing)"
)

# 2. Everyone else — own account first.
not_managed = ~is_managed_by_williams
df_subscriptions.loc[not_managed, "TargetAccountNumber"] = df_subscriptions.loc[not_managed, "AccountCode"].map(own_account_map)
df_subscriptions.loc[not_managed & df_subscriptions["TargetAccountNumber"].notna(), "TargetAccountKey"] = "own_account"

# 3. Everyone else, still without an account — Williams Corporation bucket fallback.
missing_own_account = not_managed & df_subscriptions["TargetAccountNumber"].isna()
if missing_own_account.any():
    logger.warning(
        f"{missing_own_account.sum():,} subscriptions have no matching created/existing account in "
        f"04_Create_Accounts.ipynb's results and aren't Managed-by-Williams — falling back to the "
        f"Williams Corporation bucket account for these."
    )
    df_subscriptions.loc[missing_own_account, "TargetAccountKey"] = "williams_corporation"
    df_subscriptions.loc[missing_own_account, "TargetAccountNumber"] = williams_corporation_number

logger.info(df_subscriptions["TargetAccountKey"].value_counts(dropna=False).to_string())
df_subscriptions[["SubscriptionUSN", "AccountCode", "CustomerSuppliedReference", "TargetAccountKey", "TargetAccountNumber"]].head(20)


2026-07-27 10:11:06,931 [WARNING] No successful account_results row found for: {'williams_corporation'} — falling back to the TARGET_ACCOUNTS placeholder for those. Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run.
2026-07-27 10:11:06,933 [INFO] 2 accounts available from 04_Create_Accounts.ipynb (own-account routing)


2026-07-27 10:11:06,942 [INFO] 133 subscriptions routed to the shared 'Managed by Williams' account (Reference match — overrides own-account routing)
2026-07-27 10:11:06,985 [INFO] TargetAccountKey
managed_by_williams    133
own_account            131


,SubscriptionUSN,AccountCode,CustomerSuppliedReference,TargetAccountKey,TargetAccountNumber
0,V113061451,99965692,Williams Corporation,own_account,99965692_fullbatch5
1,V113062335,99965692,None,own_account,99965692_fullbatch5
2,V113062343,99965692,None,own_account,99965692_fullbatch5
3,V113062384,99965692,WC CHCH T9,own_account,99965692_fullbatch5
4,V113062392,99965692,WC CHCH T9,own_account,99965692_fullbatch5
5,V113062400,99965692,WC CHCH T9,own_account,99965692_fullbatch5
6,V113062418,99965692,WC CHCH T9,own_account,99965692_fullbatch5
7,V113062632,99965692,WC CHCH T9,own_account,99965692_fullbatch5
8,V113062616,99965692,WC CHCH T9,own_account,99965692_fullbatch5
9,V113062962,99965692,WC CHCH T9,own_account,99965692_fullbatch5


## 4. Look up each subscription's real address + radius username (Voyager)

Two API calls per subscription, keyed on `SupplierServiceID`:
`fetch_voyager_circuit` -> `fetch_voyager_address`, wrapped by
`get_voyager_address`. Run in parallel (it's now real network calls, not
pure string parsing) via the same `ThreadPoolExecutor` pattern used in
`06_Create_Addresses.ipynb`.

Only ACTIVE subscriptions reach this point — inactive ones were split off
in step 2b and don't go through Voyager at all.

Subscriptions with no `SupplierServiceID`, or where either Voyager call
fails, come back with `parsed_ok = False` — flagged the same way unparsed
labels used to be, and skipped by `06_Create_Addresses.ipynb`.


In [20]:
if VOYAGER_CCP_KEY is None or VOYAGER_PARTNER_ID is None:
    logger.warning("VOYAGER_CCP_KEY / VOYAGER_PARTNER_ID not set — every Voyager lookup below will fail. Set them in .env.")

blank_supplier_ids = df_subscriptions["SupplierServiceID"].isna() | (df_subscriptions["SupplierServiceID"].astype(str).str.strip() == "")
if blank_supplier_ids.all():
    logger.warning(
        "SupplierServiceID is blank for EVERY subscription in this batch — the Voyager circuits lookup "
        "can't run at all without it, so every ParsedAddress_* field below will be None. Check the MySQL "
        "source data / SUBSCRIPTION_QUERY in step 2 before re-running."
    )
elif blank_supplier_ids.any():
    logger.warning(f"{blank_supplier_ids.sum():,} / {len(df_subscriptions):,} subscriptions have a blank SupplierServiceID")

voyager_session = new_voyager_session(max_workers=MAX_WORKERS)


def _lookup_row(supplier_service_id):
    return get_voyager_address(voyager_session, supplier_service_id)


logger.info(f"Looking up Voyager address for {len(df_subscriptions):,} subscriptions with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    voyager_results = list(executor.map(_lookup_row, df_subscriptions["SupplierServiceID"]))

existing_parsed_cols = [c for c in df_subscriptions.columns if c.startswith("ParsedAddress_")]
if existing_parsed_cols:
    df_subscriptions = df_subscriptions.drop(columns=existing_parsed_cols)  # safe to re-run this cell

address_parts = pd.DataFrame(voyager_results, index=df_subscriptions.index)
address_parts = address_parts.add_prefix("ParsedAddress_")
df_subscriptions = pd.concat([df_subscriptions, address_parts], axis=1)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    logger.warning(f"{len(unparsed):,} subscriptions could not be resolved to a Voyager address")
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info("Error breakdown:\n" + error_summary.to_string(index=False))

df_subscriptions[[
    "SubscriptionLabel", "SupplierServiceID",
    "ParsedAddress_addLine1", "ParsedAddress_addLine2", "ParsedAddress_city",
    "ParsedAddress_postcode", "ParsedAddress_region_iso", "ParsedAddress_region_code_raw", "ParsedAddress_radius_user",
    "ParsedAddress_parsed_ok", "ParsedAddress_error",
]].head(20)


2026-07-27 10:11:07,031 [WARNING] 3 / 264 subscriptions have a blank SupplierServiceID
2026-07-27 10:11:07,059 [INFO] Looking up Voyager address for 264 subscriptions with 10 workers...
2026-07-27 10:11:49,596 [WARNING] Voyager 429 on https://api.voyager.nz/address-search/v3/addresses/id/f2be7a4bae3af3a8f1810836416e4ca3062be72a — retry 1/6 in 29.4s
2026-07-27 10:11:50,159 [WARNING] Voyager 500 on https://api.voyager.nz/fibre/v1/circuits/1643180358 — retry 1/3 in 2.4s
2026-07-27 10:11:50,596 [WARNING] Voyager 429 on https://api.voyager.nz/address-search/v3/addresses/id/1f5b0d33d3fc6478cf7a1c2c910fd831c56d69fa — retry 1/6 in 28.0s
2026-07-27 10:11:51,153 [WARNING] Voyager 500 on https://api.voyager.nz/fibre/v1/circuits/1643180367 — retry 1/3 in 2.5s
2026-07-27 10:11:51,586 [WARNING] Voyager 429 on https://api.voyager.nz/address-search/v3/addresses/id/d6c4334b30065884470b1a4806a4f0a569702014 — retry 1/6 in 27.4s
2026-07-27 10:11:52,087 [WARNING] Voyager 429 on https://api.voyager.nz/addre

,SubscriptionLabel,SupplierServiceID,ParsedAddress_addLine1,ParsedAddress_addLine2,ParsedAddress_city,ParsedAddress_postcode,ParsedAddress_region_iso,ParsedAddress_region_code_raw,ParsedAddress_radius_user,ParsedAddress_parsed_ok,ParsedAddress_error
0,matthew.horncastle@williamsinternet.com,ENVOYB02618143,39 Chester Street West,Christchurch Central,Christchurch,8013,CAN,CAN,matthew.horncastle@williamsinternet.com,True,None
1,V113062335,None,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription
2,V113062343,None,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription
3,9matataway@williamsinternet.com,UFF000007009674,9 Matata Way,Tauranga,Tauranga,3110,BOP,BOP,9matataway@williamsinternet.com,True,None
4,3matataway@williamsinternet.com,UFF000007009635,3 Matata Way,Tauranga,Tauranga,3110,BOP,BOP,3matataway@williamsinternet.com,True,None
5,5matataway@williamsinternet.com,UFF000007009642,None,None,None,None,None,None,None,False,circuits lookup failed: HTTPSConnectionPool(ho...
6,11matataway@williamsinternet.com,UFF000007009690,None,None,None,None,None,None,None,False,circuits lookup failed: HTTPSConnectionPool(ho...
7,7.13biddle@williamsinternet.com,1643175950,None,None,None,None,None,None,None,False,circuits lookup failed: HTTPSConnectionPool(ho...
8,7matataway@williamsinternet.com,UFF000007009661,None,None,None,None,None,None,None,False,circuits lookup failed: HTTPSConnectionPool(ho...
9,1810fathom@williamsinternet.com,1643175154,None,None,None,None,None,None,None,False,circuits lookup failed: HTTPSConnectionPool(ho...


In [21]:
# Second pass: retry subscriptions that failed with a transient 5xx on the
# first pass, after a longer cooldown. 404s / "no SupplierServiceID" are
# permanent and intentionally skipped -- see voyager_second_pass() docstring.
SECOND_PASS_DELAY_SECONDS = 90  # bump this up if the same circuits still fail after 90s

df_subscriptions = voyager_second_pass(
    df_subscriptions,
    voyager_session,
    delay_seconds=SECOND_PASS_DELAY_SECONDS,
    max_workers=MAX_WORKERS,
)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info(f"{len(unparsed):,} subscriptions still unresolved overall:\n" + error_summary.to_string(index=False))
else:
    logger.info("All subscriptions resolved.")

unparsed[["SubscriptionUSN", "SupplierServiceID", "ParsedAddress_error"]]


2026-07-27 10:19:26,038 [INFO] Voyager second pass: 24 subscriptions failed with a transient 5xx on the first pass — waiting 90s before retrying...
2026-07-27 10:21:19,576 [INFO] Voyager second pass: 23 / 24 resolved on retry.
2026-07-27 10:21:19,581 [INFO] 17 subscriptions still unresolved after second pass:
                                                                                                                                                                                                                                                                          error  count
                                                                                                                                                                                                                                      no SupplierServiceID on this subscription      3
                                                                                                                                   

,SubscriptionUSN,SupplierServiceID,ParsedAddress_error
1,V113062335,None,no SupplierServiceID on this subscription
2,V113062343,None,no SupplierServiceID on this subscription
5,V113062400,UFF000007009642,circuits lookup failed: HTTPSConnectionPool(ho...
6,V113062418,UFF000007009690,circuits lookup failed: HTTPSConnectionPool(ho...
7,V113062632,1643175950,circuits lookup failed: HTTPSConnectionPool(ho...
8,V113062616,UFF000007009661,circuits lookup failed: HTTPSConnectionPool(ho...
9,V113062962,1643175154,circuits lookup failed: HTTPSConnectionPool(ho...
10,V113062970,1643175163,circuits lookup failed: HTTPSConnectionPool(ho...
11,V113062988,1643175168,circuits lookup failed: HTTPSConnectionPool(ho...
43,V113064133,1643180358,circuits lookup failed: ('Connection aborted.'...


In [22]:
df_subscriptions["SupplierServiceID"] = df_subscriptions["SupplierServiceID"] + BATCH_NUMBER
df_subscriptions["ParsedAddress_radius_user	"] = df_subscriptions["ParsedAddress_radius_user"] + BATCH_NUMBER
df_subscriptions.head()

,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,ParsedAddress_city,ParsedAddress_postcode,ParsedAddress_region_name,ParsedAddress_region_iso,ParsedAddress_region_code_raw,ParsedAddress_radius_user,ParsedAddress_location_id,ParsedAddress_parsed_ok,ParsedAddress_error,ParsedAddress_radius_user\t
0,4163350815,2026-07-03 20:36:41,260726,vBill,99965692,Broadband - Fibre,V113061451,matthew.horncastle@williamsinternet.com,2025-11-26,None,...,Christchurch,8013,CANTERBURY REGION,CAN,CAN,matthew.horncastle@williamsinternet.com,36049ea46dd36a68eba30fd8cd7f6012a7d7e650,True,None,matthew.horncastle@williamsinternet.com_fullba...
1,4167925657,2025-12-02 01:12:40,260841,vBill,99965692,Platforms,V113062335,V113062335,2025-12-01,None,...,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription,NaN
2,4167926822,2025-12-02 01:12:40,260842,vBill,99965692,Platforms,V113062343,V113062343,2025-12-01,None,...,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription,NaN
3,4167997507,2026-07-03 20:36:54,260846,vBill,99965692,Broadband - Fibre,V113062384,9matataway@williamsinternet.com,2025-12-01,None,...,Tauranga,3110,BAY OF PLENTY REGION,BOP,BOP,9matataway@williamsinternet.com,44d46826807ccd9435dd3bc7070281ba2afb4cb2,True,None,9matataway@williamsinternet.com_fullbatch5
4,4167998688,2026-07-03 20:36:54,260847,vBill,99965692,Broadband - Fibre,V113062392,3matataway@williamsinternet.com,2025-12-01,None,...,Tauranga,3110,BAY OF PLENTY REGION,BOP,BOP,3matataway@williamsinternet.com,9160e43272e2e61bc9c00f45651914d21dab5a84,True,None,3matataway@williamsinternet.com_fullbatch5


## 5. Save

In [23]:
save_df("subscriptions_resolved", df_subscriptions)


Saved 264 rows -> migration_data\05_subscriptions_resolved.csv
